In [1]:
from pathlib import Path

base_path = Path("car_damage/car_damage_yolo")

label_dirs = [
    base_path / "labels/train",
    base_path / "labels/val"
]

image_dirs = {
    "train": base_path / "images/train",
    "val": base_path / "images/val"
}

In [2]:
bad_files = []

for label_dir in label_dirs:
    split = label_dir.parts[-1]  # train / val
    img_dir = image_dirs[split]

    for txt_file in label_dir.glob("*.txt"):
        with open(txt_file, "r") as f:
            lines = f.readlines()

        for line_no, line in enumerate(lines, start=1):
            parts = line.strip().split()

            # check format
            if len(parts) != 5:
                bad_files.append((txt_file, line_no, "Invalid format", line.strip()))
                continue

            cls = parts[0]

            # ❌ wrong class id
            if cls != "0":
                bad_files.append((txt_file, line_no, "Wrong class", line.strip()))

            # ❌ invalid coordinates
            try:
                coords = list(map(float, parts[1:]))
                if not all(0.0 <= x <= 1.0 for x in coords):
                    bad_files.append((txt_file, line_no, "Invalid coords", line.strip()))
            except:
                bad_files.append((txt_file, line_no, "Parse error", line.strip()))

In [4]:
print(f"Total issues found: {len(bad_files)}\n")

for item in bad_files[:50]:  # show first 50
    txt_file, line_no, issue, content = item

    split = txt_file.parts[-2]
    img_name = txt_file.stem

    img_path = image_dirs[split] / f"{img_name}.jpg"

    if not img_path.exists():
        img_path = image_dirs[split] / f"{img_name}.jpeg"
    if not img_path.exists():
        img_path = image_dirs[split] / f"{img_name}.png"

    print(f"""
Label file : {txt_file}
Image file : {img_path}
Line       : {line_no}
Issue      : {issue}
Content    : {content}
""")

Total issues found: 0

